# Feature Stores with Feast

Builds a tiny local feature repo programmatically (SQLite online store, parquet
offline store), then exercises both retrieval paths. Requires `pip install feast pandas pyarrow`.
Feast's API moves between versions — if a cell errors, check `feast version` against the
docs; the concepts are stable even when names shift.

## 1. Generate a source dataset with timestamps (the offline store)

In [ ]:
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
now = datetime.utcnow()

rows = []
for driver_id in [1001, 1002, 1003]:
    for hours_back in range(72, 0, -1):           # 3 days of hourly feature values
        rows.append({
            "driver_id": driver_id,
            "event_timestamp": now - timedelta(hours=hours_back),
            "created": now,
            "conv_rate": float(rng.uniform(0.1, 0.9)),
            "avg_daily_trips": int(rng.integers(5, 40)),
        })

source_df = pd.DataFrame(rows)
import os
os.makedirs("feature_repo/data", exist_ok=True)
source_df.to_parquet("feature_repo/data/driver_stats.parquet", index=False)
source_df.tail(3)

## 2. Define the feature repo: config, entity, feature view

In [ ]:
config = (
    "project: learning_repo\n"
    "registry: data/registry.db\n"
    "provider: local\n"
    "online_store:\n"
    "  type: sqlite\n"
    "  path: data/online_store.db\n"
    "entity_key_serialization_version: 3\n"
)
with open("feature_repo/feature_store.yaml", "w") as f:
    f.write(config)

from datetime import timedelta as td

from feast import Entity, FeatureStore, FeatureView, Field, FileSource
from feast.types import Float32, Int64

driver = Entity(name="driver", join_keys=["driver_id"])

driver_stats_source = FileSource(
    path="data/driver_stats.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created",
)

driver_stats_fv = FeatureView(
    name="driver_hourly_stats",
    entities=[driver],
    ttl=td(days=2),
    schema=[Field(name="conv_rate", dtype=Float32),
            Field(name="avg_daily_trips", dtype=Int64)],
    source=driver_stats_source,
)

store = FeatureStore(repo_path="feature_repo")
store.apply([driver, driver_stats_fv])
print("applied:", [fv.name for fv in store.list_feature_views()])

## 3. Materialize: push latest values into the online store

In [ ]:
store.materialize_incremental(end_date=datetime.utcnow())
print("online store populated with latest value per driver")

## 4. Online retrieval (what serving does, in milliseconds)

In [ ]:
online = store.get_online_features(
    features=["driver_hourly_stats:conv_rate",
              "driver_hourly_stats:avg_daily_trips"],
    entity_rows=[{"driver_id": 1001}, {"driver_id": 1002}],
).to_dict()
pd.DataFrame(online)

## 5. Point-in-time historical retrieval (what training does)

In [ ]:
# Pretend these are label events: each row needs features AS OF its own timestamp.
entity_df = pd.DataFrame({
    "driver_id": [1001, 1001, 1002, 1003],
    "event_timestamp": [now - timedelta(hours=40), now - timedelta(hours=2),
                        now - timedelta(hours=20), now - timedelta(hours=1)],
})

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=["driver_hourly_stats:conv_rate",
              "driver_hourly_stats:avg_daily_trips"],
).to_df()
training_df

## 6. Mini-project: compare with a naive (leaky) join

In [ ]:
# The naive approach: just grab each driver's LATEST value — leaking the future
# into rows whose label event happened long before that value existed.
naive = source_df.sort_values("event_timestamp").groupby("driver_id").last().reset_index()
leaky = entity_df.merge(naive[["driver_id", "conv_rate"]], on="driver_id")

comparison = training_df[["driver_id", "event_timestamp", "conv_rate"]].merge(
    leaky, on="driver_id", suffixes=("_point_in_time", "_naive_leaky"))
comparison
# Rows where the two conv_rate columns differ are exactly the leakage a
# point-in-time join prevents.